
# Pub/Sub Hands-On Lab

Publisher/subscriber systems are everywhere once you start looking — every time you place a
Zomato order, swipe a credit card, or book a ticket, there's one running under the hood in some
form.

The concrete problem: an order comes in, and inventory, billing, analytics, and fraud detection all
need to know *right now* — without the order service knowing or caring who's listening, or waiting
around for any of them to finish. That's what Pub/Sub is for: publish once, every subscriber gets
its own full copy, at its own pace, and delivery survives a subscriber being down for a while.

Auth uses the `gcloud` CLI (Colab's sign-in flow); everything else goes through the
`google-cloud-pubsub` Python client, not the CLI.

Run top to bottom. You'll need `roles/pubsub.editor` (or better), and the Pub/Sub API enabled
(a cell below does that).


## 1. Auth

In [ ]:

import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import auth
    auth.authenticate_user()
    print("authenticated")
else:
    print("not in Colab — assuming ADC is already set up")


## 2. Install & config

In [ ]:
!pip install --quiet "google-cloud-pubsub>=2.18.0"

In [ ]:

import time, json
from google.cloud import pubsub_v1
from google.pubsub_v1.types import Schema, Encoding
from google.api_core import exceptions as gcs_exceptions
from google.api_core.retry import Retry
from google.protobuf import duration_pb2


In [ ]:

PROJECT_ID = input("Enter your GCP Project ID: ").strip()

!gcloud config set project {PROJECT_ID}
!gcloud services enable pubsub.googleapis.com cloudfunctions.googleapis.com \
    cloudbuild.googleapis.com run.googleapis.com artifactregistry.googleapis.com \
    --project={PROJECT_ID}

_suffix = int(time.time())
TOPIC_ID = f"training-topic-{_suffix}"
SCHEMA_TOPIC_ID = f"training-schema-topic-{_suffix}"
SCHEMA_ID = f"training-schema-{_suffix}"
SUB_A_ID = f"training-sub-a-{_suffix}"
SUB_B_ID = f"training-sub-b-{_suffix}"
ORDERED_SUB_ID = f"training-sub-ordered-{_suffix}"
FILTERED_SUB_ID = f"training-sub-filtered-{_suffix}"
EXACTLY_ONCE_SUB_ID = f"training-sub-exactly-once-{_suffix}"
DLQ_TOPIC_ID = f"training-dlq-topic-{_suffix}"
DLQ_SOURCE_SUB_ID = f"training-sub-with-dlq-{_suffix}"
SNAPSHOT_ID = f"training-snapshot-{_suffix}"

publisher = pubsub_v1.PublisherClient()
subscriber = pubsub_v1.SubscriberClient()
print("ready, main topic will be", TOPIC_ID)


## 3. Topics

In [ ]:

topic_path = publisher.topic_path(PROJECT_ID, TOPIC_ID)
topic = publisher.create_topic(request={"name": topic_path})

for t in publisher.list_topics(request={"project": f"projects/{PROJECT_ID}"}):
    print(" -", t.name)

fetched = publisher.get_topic(request={"topic": topic_path})
print("\nlabels:", dict(fetched.labels))



> 🖥️ Pub/Sub > Topics — should be listed. Every subscription and publish call from here on
> attaches to this one.



## 4. Schema Registry
A schema lets a topic reject messages that don't match an agreed shape at publish time — catch a
bad payload here instead of discovering it three steps downstream in BigQuery.

(Schema support has shifted across `google-cloud-pubsub` versions — if a call below looks off
compared to current docs, check your installed version first.)


In [ ]:

schema_client = pubsub_v1.SchemaServiceClient()
schema_path = schema_client.schema_path(PROJECT_ID, SCHEMA_ID)

avro_definition = json.dumps({
    "type": "record", "name": "TrainingEvent",
    "fields": [
        {"name": "event_id", "type": "string"},
        {"name": "event_type", "type": "string"},
        {"name": "amount", "type": "double"},
    ],
})

schema = schema_client.create_schema(request={
    "parent": f"projects/{PROJECT_ID}",
    "schema": {"name": schema_path, "type_": Schema.Type.AVRO, "definition": avro_definition},
    "schema_id": SCHEMA_ID,
})

schema_topic_path = publisher.topic_path(PROJECT_ID, SCHEMA_TOPIC_ID)
schema_topic = publisher.create_topic(request={
    "name": schema_topic_path,
    "schema_settings": {"schema": schema_path, "encoding": Encoding.JSON},
})
print("schema + enforced topic both created")


In [ ]:

valid = json.dumps({"event_id": "evt-001", "event_type": "purchase", "amount": 42.5}).encode()
publisher.publish(schema_topic_path, data=valid).result()
print("valid message went through fine")

invalid = json.dumps({"event_id": "evt-002", "amount": "not-a-number"}).encode()
try:
    publisher.publish(schema_topic_path, data=invalid).result()
    print("uh, that shouldn't have worked — check schema enforcement")
except gcs_exceptions.GoogleAPICallError as e:
    print("rejected as expected:", str(e)[:150])



> 🖥️ Pub/Sub > Schemas shows the Avro definition; the schema topic's Schema tab shows it attached
> with JSON encoding.



## Architecture, briefly

```
                                   ┌─────────────────┐
                                   │   Subscription A │──── pull() ────▶ Consumer A
                              ┌───▶│  (own ack state)  │
   Publisher ──publish()──▶ Topic  └─────────────────┘
                              │    ┌─────────────────┐
                              └───▶│   Subscription B │──push──▶ HTTPS endpoint
                                   │  (own ack state)  │        (called by Pub/Sub)
                                   └─────────────────┘
```

Three things worth internalizing before the rest of this makes sense:

- **Fan-out is free.** Every subscription gets its own complete copy. Acking on one does nothing
  to another's copy.
- **Delivery is at-least-once**, not exactly-once, by default (Section 14 covers the opt-in for
  the latter). Rare redelivery after a successful ack is possible — design for idempotency.
- **The ack deadline *is* the retry mechanism.** An unacked message becomes redeliverable once its
  deadline passes, or immediately via an explicit nack. There's no separate retry system bolted on.

Pull vs push is the same subscription concept with two different delivery mechanics: pull means
*your* code calls `subscriber.pull()`; push means Pub/Sub calls an HTTPS endpoint *you* stood up,
and treats any non-2xx response (or a timeout) as a nack.



## 5. Subscriptions — pull, push, fan-out

Two pull subscriptions on the same topic first — this is fan-out in its simplest form.


In [ ]:

sub_a_path = subscriber.subscription_path(PROJECT_ID, SUB_A_ID)
sub_b_path = subscriber.subscription_path(PROJECT_ID, SUB_B_ID)

subscriber.create_subscription(request={"name": sub_a_path, "topic": topic_path})
subscriber.create_subscription(request={"name": sub_b_path, "topic": topic_path})
print("A and B created, both independent, both attached to the same topic")



> 🖥️ Pub/Sub > Subscriptions — both listed against the same parent topic.

### Push, for real
Push subscriptions usually get demoed as configuration syntax only, since they need a real HTTPS
endpoint most training notebooks don't have lying around. So let's actually deploy one — a tiny
Cloud Function (2nd gen) — and watch a push genuinely land, not just trust that it would.

Why Cloud Functions specifically: Pub/Sub auto-trusts `*.cloudfunctions.net`/`*.run.app` URLs for
push, no domain-ownership dance required.


In [ ]:

import os
os.makedirs("push_function", exist_ok=True)

with open("push_function/main.py", "w") as f:
    f.write('''
import base64
import functions_framework

@functions_framework.http
def pubsub_push_handler(request):
    envelope = request.get_json()
    if not envelope or "message" not in envelope:
        return ("Bad Request: no Pub/Sub message received", 400)
    message = envelope["message"]
    data = base64.b64decode(message.get("data", "")).decode("utf-8") if message.get("data") else ""
    print(f"Received push message: data={data!r} attributes={message.get('attributes', {})}")
    return ("", 204)
''')

with open("push_function/requirements.txt", "w") as f:
    f.write("functions-framework==3.*\n")

print("push_function/ written")



`--no-allow-unauthenticated` keeps it private — Pub/Sub authenticates via a service account + OIDC
token, which is the actual production-correct way to secure a push endpoint.


In [ ]:

PUSH_FUNCTION_NAME = f"training-push-handler-{_suffix}"
REGION = REGION if 'REGION' in dir() else 'us-central1'

# Newer projects don't auto-grant the Compute Engine default service account the Editor role
# anymore (a 2024+ policy change), which Cloud Functions gen2 builds rely on by default.
# Without this, the build step fails with "missing permission on the build service account."
project_number = !gcloud projects describe {PROJECT_ID} --format="value(projectNumber)"
project_number = project_number[0].strip()
compute_sa = f"{project_number}-compute@developer.gserviceaccount.com"

!gcloud projects add-iam-policy-binding {PROJECT_ID} \
    --member="serviceAccount:{compute_sa}" \
    --role="roles/cloudbuild.builds.builder" --quiet

print("granted Cloud Build Service Account role to", compute_sa, "-- waiting for it to propagate...")
time.sleep(30)

!gcloud functions deploy {PUSH_FUNCTION_NAME} --gen2 --runtime=python312 --region={REGION} \
    --source=push_function --entry-point=pubsub_push_handler --trigger-http \
    --no-allow-unauthenticated --project={PROJECT_ID} --quiet

push_url = !gcloud functions describe {PUSH_FUNCTION_NAME} --gen2 --region={REGION} \
    --project={PROJECT_ID} --format="value(serviceConfig.uri)"
PUSH_ENDPOINT_URL = push_url[0].strip()
print("deployed at", PUSH_ENDPOINT_URL)


In [ ]:

# dedicated service account so Pub/Sub can invoke the function, plus the push subscription itself
PUSH_SA_NAME = f"push-inv-{_suffix}"
PUSH_SA_EMAIL = f"{PUSH_SA_NAME}@{PROJECT_ID}.iam.gserviceaccount.com"

!gcloud iam service-accounts create {PUSH_SA_NAME} --display-name="Pub/Sub push invoker" --project={PROJECT_ID} --quiet
!gcloud functions add-invoker-policy-binding {PUSH_FUNCTION_NAME} --gen2 --region={REGION} \
    --member="serviceAccount:{PUSH_SA_EMAIL}" --project={PROJECT_ID} --quiet

push_sub_id = f"training-sub-push-{_suffix}"
push_sub_path = subscriber.subscription_path(PROJECT_ID, push_sub_id)
subscriber.create_subscription(request={
    "name": push_sub_path, "topic": topic_path,
    "push_config": {"push_endpoint": PUSH_ENDPOINT_URL, "oidc_token": {"service_account_email": PUSH_SA_EMAIL}},
})
print("push subscription wired up to", PUSH_ENDPOINT_URL)


In [ ]:

# IAM can take a few seconds to catch up -- if this shows an auth error in the logs, just re-run once
publisher.publish(topic_path, data=b"pushed via a real Cloud Function!", demo="push").result()
print("published, waiting on delivery + log propagation...")
time.sleep(15)

!gcloud functions logs read {PUSH_FUNCTION_NAME} --gen2 --region={REGION} --project={PROJECT_ID} --limit=10



Look for `Received push message: data='pushed via a real Cloud Function!'` in the log output — that's
Pub/Sub itself calling your endpoint, not anything simulated locally.

> 🖥️ The push subscription's Delivery type shows Push with the function URL attached. Cloud Run
> (gen2 functions run on Cloud Run under the hood) shows the same log line in its own Logs tab.



## 6. Publishing
Single message, a batch, and ordered delivery — the three shapes you'll actually reach for.


In [ ]:

future = publisher.publish(topic_path, data=b"Hello from the lab!", source="training-notebook", priority="normal")
print("message id:", future.result())

futures = [publisher.publish(topic_path, data=f"batch {i}".encode(), index=str(i)) for i in range(10)]
ids = [f.result() for f in futures]
print(f"published {len(ids)} more, batched under the hood")



> 🖥️ Topic's Metrics tab — publish count jumps by 11. It won't show you "1 call vs 10 calls,"
> since batching is purely a client-side efficiency, not a different wire format.


In [ ]:

# ordering: only messages sharing the same ordering_key are delivered in publish order.
# Pub/Sub gives NO ordering guarantee otherwise. Needs enabling on both sides.
ordered_publisher = pubsub_v1.PublisherClient(
    publisher_options=pubsub_v1.types.PublisherOptions(enable_message_ordering=True)
)
ordered_sub_path = subscriber.subscription_path(PROJECT_ID, ORDERED_SUB_ID)
subscriber.create_subscription(request={"name": ordered_sub_path, "topic": topic_path, "enable_message_ordering": True})

for i in range(5):
    ordered_publisher.publish(topic_path, data=f"ordered event {i}".encode(), ordering_key="order-key-1")
print("5 ordered messages published under order-key-1")



## 7. Pull, ack, nack


In [ ]:

response = subscriber.pull(request={"subscription": sub_a_path, "max_messages": 10}, retry=Retry(deadline=30))
print(f"pulled {len(response.received_messages)} on A")
for rm in response.received_messages:
    print(" -", rm.message.data[:60])


In [ ]:

# acked messages won't come back to A. B still has its own untouched copies (next cell).
ack_ids = [rm.ack_id for rm in response.received_messages]
if ack_ids:
    subscriber.acknowledge(request={"subscription": sub_a_path, "ack_ids": ack_ids})
    print(f"acked {len(ack_ids)}")
else:
    print("nothing pulled — publish more first")



> 🖥️ Subscriber A's Unacked message count drops to 0 right after — proof the ack registered
> server-side, not just in this notebook's variables.


In [ ]:

# nack = reset ack deadline to 0, forcing immediate redelivery instead of waiting it out
publisher.publish(topic_path, data=b"message to be nacked").result()
time.sleep(2)

resp = subscriber.pull(request={"subscription": sub_a_path, "max_messages": 1})
if resp.received_messages:
    nack_id = resp.received_messages[0].ack_id
    subscriber.modify_ack_deadline(request={"subscription": sub_a_path, "ack_ids": [nack_id], "ack_deadline_seconds": 0})
    print("nacked — should come right back")
else:
    print("nothing pulled — may already be consumed")


In [ ]:

# B never had anything acked -- everything from Section 6 should still be sitting here, untouched
response_b = subscriber.pull(request={"subscription": sub_b_path, "max_messages": 20})
print(f"B still has {len(response_b.received_messages)} pending, independent of whatever happened to A")

ack_ids_b = [rm.ack_id for rm in response_b.received_messages]
if ack_ids_b:
    subscriber.acknowledge(request={"subscription": sub_b_path, "ack_ids": ack_ids_b})



> 🖥️ Check B's Unacked count *before* running that last cell — it should show the full backlog.
> That's fan-out working: A acking its copy never touched B's.



## 8. Streaming pull
The pattern real consumers actually use instead of polling `pull()` in a loop — open a stream,
get a callback per message. This runs ~15s: publishes a few messages partway through, then shuts
down cleanly.


In [ ]:

received = []

def callback(message):
    received.append(message.data)
    print("got:", message.data[:60])
    message.ack()

streaming_future = subscriber.subscribe(sub_a_path, callback=callback)
time.sleep(3)
for i in range(3):
    publisher.publish(topic_path, data=f"streamed {i}".encode()).result()
time.sleep(12)

streaming_future.cancel()
try:
    streaming_future.result(timeout=5)
except Exception:
    pass

print(f"received {len(received)} via streaming pull")



## 9. Dead-letter topics
The gotcha here isn't the concept, it's the IAM: the Pub/Sub service agent needs explicit
permission to publish to your dead-letter topic *and* to subscribe on the source subscription, or
dead-lettering silently just doesn't fire. Easy to forget both.


In [ ]:

dlq_topic_path = publisher.topic_path(PROJECT_ID, DLQ_TOPIC_ID)
publisher.create_topic(request={"name": dlq_topic_path})

project_number = !gcloud projects describe {PROJECT_ID} --format="value(projectNumber)"
project_number = project_number[0].strip()
pubsub_service_agent = f"serviceAccount:service-{project_number}@gcp-sa-pubsub.iam.gserviceaccount.com"

!gcloud pubsub topics add-iam-policy-binding {DLQ_TOPIC_ID} --project={PROJECT_ID} \
    --member="{pubsub_service_agent}" --role="roles/pubsub.publisher"

print("dead-letter topic created, service agent granted publish")


In [ ]:

# after 5 failed delivery attempts, forward to the DLQ instead of retrying forever
dlq_source_sub_path = subscriber.subscription_path(PROJECT_ID, DLQ_SOURCE_SUB_ID)
subscriber.create_subscription(request={
    "name": dlq_source_sub_path, "topic": topic_path,
    "dead_letter_policy": {"dead_letter_topic": dlq_topic_path, "max_delivery_attempts": 5},
})

!gcloud pubsub subscriptions add-iam-policy-binding {DLQ_SOURCE_SUB_ID} --project={PROJECT_ID} \
    --member="{pubsub_service_agent}" --role="roles/pubsub.subscriber"

print("dlq-backed subscription ready")



> 🖥️ Dead-letter topic's Permissions tab shows the service agent with Publisher. Source
> subscription shows "Dead lettering" with max attempts: 5.

Now force it — nack the same message repeatedly until Pub/Sub gives up and forwards it. Takes a
minute or two.


In [ ]:

publisher.publish(topic_path, data=b"this message will be dead-lettered").result()

for attempt in range(6):
    time.sleep(5)
    resp = subscriber.pull(request={"subscription": dlq_source_sub_path, "max_messages": 1})
    if not resp.received_messages:
        continue
    ack_id = resp.received_messages[0].ack_id
    subscriber.modify_ack_deadline(request={"subscription": dlq_source_sub_path, "ack_ids": [ack_id], "ack_deadline_seconds": 0})
    print(f"attempt {attempt}: nacked again")

# check the DLQ topic directly
dlq_check = subscriber.subscription_path(PROJECT_ID, f"{DLQ_SOURCE_SUB_ID}-dlq-check")
subscriber.create_subscription(request={"name": dlq_check, "topic": dlq_topic_path})
dlq_resp = subscriber.pull(request={"subscription": dlq_check, "max_messages": 5})
print(f"\n{len(dlq_resp.received_messages)} message(s) on the dead-letter topic")
for rm in dlq_resp.received_messages:
    print(" -", rm.message.data)
    subscriber.acknowledge(request={"subscription": dlq_check, "ack_ids": [rm.ack_id]})



## 10. Snapshots & replay
A snapshot freezes a subscription's ack state. Seeking back to it later makes everything that was
unacked (or published after) redeliverable again — a way to "rewind" after finding a bug in your
consumer.


In [ ]:

snapshot_path = subscriber.snapshot_path(PROJECT_ID, SNAPSHOT_ID)
subscriber.create_snapshot(request={"name": snapshot_path, "subscription": sub_a_path})
print("snapshot taken")



> 🖥️ Pub/Sub > Snapshots — listed, with its source subscription and expiry (7 days by default).


In [ ]:

for i in range(3):
    publisher.publish(topic_path, data=f"post-snapshot {i}".encode()).result()
time.sleep(2)

# drain everything first so the replay below is unambiguous
drain = subscriber.pull(request={"subscription": sub_a_path, "max_messages": 50})
if drain.received_messages:
    subscriber.acknowledge(request={"subscription": sub_a_path, "ack_ids": [rm.ack_id for rm in drain.received_messages]})

subscriber.seek(request={"subscription": sub_a_path, "snapshot": snapshot_path})
replayed = subscriber.pull(request={"subscription": sub_a_path, "max_messages": 50})
print(f"replayed {len(replayed.received_messages)} message(s) after seeking back")
if replayed.received_messages:
    subscriber.acknowledge(request={"subscription": sub_a_path, "ack_ids": [rm.ack_id for rm in replayed.received_messages]})


In [ ]:

# seek also takes a raw timestamp -- "replay the last 10 minutes" without a pre-made snapshot
from google.protobuf.timestamp_pb2 import Timestamp

ten_min_ago = Timestamp()
ten_min_ago.FromSeconds(int(time.time()) - 600)
subscriber.seek(request={"subscription": sub_a_path, "time": ten_min_ago})
print("sought back 10 minutes")



## 11. Flow control & retry policy


In [ ]:

# caps how much a slow subscriber can have in flight at once, so a fast publisher can't overwhelm it
flow_control = pubsub_v1.types.FlowControl(max_messages=5, max_bytes=1024 * 1024)
limited = []

def limited_callback(message):
    limited.append(message.data)
    time.sleep(1)  # pretend this is slow
    message.ack()

for i in range(8):
    publisher.publish(topic_path, data=f"flow-control {i}".encode()).result()

flow_future = subscriber.subscribe(sub_b_path, callback=limited_callback, flow_control=flow_control)
time.sleep(8)
flow_future.cancel()
try:
    flow_future.result(timeout=5)
except Exception:
    pass
print(f"processed {len(limited)} under flow control (max 5 in flight)")


In [ ]:

# server-side backoff timing for redelivery -- no client code needed to benefit
retry_sub_id = f"training-sub-retry-{_suffix}"
retry_sub_path = subscriber.subscription_path(PROJECT_ID, retry_sub_id)
subscriber.create_subscription(request={
    "name": retry_sub_path, "topic": topic_path,
    "retry_policy": {
        "minimum_backoff": duration_pb2.Duration(seconds=10),
        "maximum_backoff": duration_pb2.Duration(seconds=600),
    },
})
print("retry policy set: 10s-600s backoff")



## 12. Filtering
A subscription can filter on attributes server-side — non-matching messages get auto-acked before
your code ever sees them. Cheaper than pulling everything and filtering yourself.


In [ ]:

filtered_sub_path = subscriber.subscription_path(PROJECT_ID, FILTERED_SUB_ID)
subscriber.create_subscription(request={"name": filtered_sub_path, "topic": topic_path, "filter": 'attributes.priority = "high"'})

publisher.publish(topic_path, data=b"low priority event", priority="low").result()
publisher.publish(topic_path, data=b"high priority event", priority="high").result()
time.sleep(2)

resp = subscriber.pull(request={"subscription": filtered_sub_path, "max_messages": 10})
print(f"got {len(resp.received_messages)} (expect just the high-priority one)")
for rm in resp.received_messages:
    print(" -", rm.message.data, rm.message.attributes.get("priority"))
if resp.received_messages:
    subscriber.acknowledge(request={"subscription": filtered_sub_path, "ack_ids": [rm.ack_id for rm in resp.received_messages]})



## 13. Exactly-once delivery
Default Pub/Sub is at-least-once — rare redelivery even after a successful ack. Turning on
exactly-once means a successful ack response is an actual guarantee, not a best-effort signal.

Worth a caveat: this is a newer subscription feature with its own throughput trade-offs — check
current docs before reaching for it on a high-volume pipeline.


In [ ]:

exactly_once_sub_path = subscriber.subscription_path(PROJECT_ID, EXACTLY_ONCE_SUB_ID)
subscriber.create_subscription(request={"name": exactly_once_sub_path, "topic": topic_path, "enable_exactly_once_delivery": True})

publisher.publish(topic_path, data=b"exactly-once demo").result()
time.sleep(2)

resp = subscriber.pull(request={"subscription": exactly_once_sub_path, "max_messages": 1})
if resp.received_messages:
    subscriber.acknowledge(request={"subscription": exactly_once_sub_path, "ack_ids": [resp.received_messages[0].ack_id]})
    print("acked — with exactly-once on, this ack is a real guarantee")
else:
    print("nothing pulled yet, try again")



## Cleanup
Everything, in order: subscriptions, snapshot, topics, schema, then the push Cloud Function and
its service account.


In [ ]:

all_subs = [
    SUB_A_ID, SUB_B_ID, ORDERED_SUB_ID, FILTERED_SUB_ID, EXACTLY_ONCE_SUB_ID,
    DLQ_SOURCE_SUB_ID, f"{DLQ_SOURCE_SUB_ID}-dlq-check", f"training-sub-retry-{_suffix}",
    f"training-sub-push-{_suffix}",
]
for sub_id in all_subs:
    try:
        subscriber.delete_subscription(request={"subscription": subscriber.subscription_path(PROJECT_ID, sub_id)})
    except gcs_exceptions.NotFound:
        pass

subscriber.delete_snapshot(request={"snapshot": snapshot_path})

for t in [topic_path, schema_topic_path, dlq_topic_path]:
    publisher.delete_topic(request={"topic": t})

schema_client.delete_schema(request={"name": schema_path})
print("subscriptions, snapshot, topics, schema — all gone")


In [ ]:

try:
    !gcloud functions delete {PUSH_FUNCTION_NAME} --gen2 --region={REGION} --project={PROJECT_ID} --quiet
    !gcloud iam service-accounts delete {PUSH_SA_EMAIL} --project={PROJECT_ID} --quiet
    print("push function + its service account cleaned up too")
except NameError:
    print("push section never ran this session, nothing to clean up there")


In [ ]:

try:
    publisher.get_topic(request={"topic": topic_path})
    print("still exists — cleanup may not have finished")
except gcs_exceptions.NotFound:
    print("confirmed gone")
